# MLflow Examples for Darts

This notebook walks through a practical MLOps workflow with Darts' native MLflow integration: enable autolog, compare forecasting models, promote the best one to the registry, and run inference from a production alias.

If you are new to Darts, see the [Quickstart Guide](https://unit8co.github.io/darts/quickstart/00-quickstart.html) first.

**Prerequisites:** install MLflow as an optional dependency:

```bash
pip install "mlflow>=3.0"
```

API reference: [darts.utils.mlflow](https://unit8co.github.io/darts/generated_api/darts.utils.mlflow.html).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import tempfile
import warnings

import mlflow
from mlflow import MlflowClient

import darts.metrics as metrics
from darts import set_option
from darts.datasets import AirPassengersDataset
from darts.models import LinearRegressionModel, RandomForestModel
from darts.models.forecasting.forecasting_model import GlobalForecastingModel
from darts.utils.mlflow import autolog, load_model

warnings.filterwarnings("ignore", category=FutureWarning)
set_option("plotting.use_darts_style", True)

PLOTLY_KWARGS = dict(
    width=800,
    height=400,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
)

## 1. MLflow setup

Point MLflow at a tracking backend and create an experiment. We use a temporary SQLite database so this notebook runs self-contained; in production, set `tracking_uri` to your team's MLflow server or local database.

In [3]:
tmpdir = tempfile.mkdtemp()
mlflow_db = os.path.join(tmpdir, "mlflow.db")

EXPERIMENT_NAME = "darts-mlflow-examples"
mlflow.set_tracking_uri(f"sqlite:///{mlflow_db}")
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('darts-mlflow-examples').name}")
print(
    f"\nTo explore runs in the UI:\n  mlflow ui --backend-store-uri sqlite:///{mlflow_db}"
)

2026/09/03 16:10:25 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/03 16:10:25 INFO mlflow.store.db.utils: Updating database tables
2026/09/03 16:10:26 INFO mlflow.tracking.fluent: Experiment with name 'darts-mlflow-examples' does not exist. Creating a new experiment.


Tracking URI: sqlite:////var/folders/4h/l09drklx06g8022q7khgyxyr0000gn/T/tmps5wn0e4a/mlflow.db
Experiment: darts-mlflow-examples

To explore runs in the UI:
  mlflow ui --backend-store-uri sqlite:////var/folders/4h/l09drklx06g8022q7khgyxyr0000gn/T/tmps5wn0e4a/mlflow.db


## 2. Load data

We use the classic AirPassengers dataset. The last 36 months are held-out for evaluation using a rolling backtest.

In [4]:
FORECAST_HORIZON = 12
BACKTEST_STRIDE = 12

series = AirPassengersDataset().load()
train, val = series[: -3 * FORECAST_HORIZON], series[-3 * FORECAST_HORIZON :]

print(f"Training: {len(train)} points | Validation: {len(val)} points")

fig = series.plotly()
fig.add_vline(
    x=train.end_time(),
    line_color="blue",
    line_dash="dash",
    annotation_text="Train / val split ",
    annotation_position="top left",
)
fig.update_layout(title="Air Passengers", **PLOTLY_KWARGS)

Training: 108 points | Validation: 36 points


## 3. Enable autolog

One line turns on automatic experiment tracking for Darts models and metrics. We enable model logging (for the registry workflow) and backtest aggregate metrics (scalar `backtest_agg_*` keys for easy run comparison).

In [5]:
autolog(log_models=True, log_backtest_aggregate=True)

<details>
<summary><strong>What does autolog capture?</strong> (click to expand)</summary>

| Trigger | Logged automatically |
|---|---|
| `model.fit(...)` | Model tags, hyperparameters, input series info, and trained model artifact when `log_models=True` |
| Standalone metric calls | Time-aggregated scalar metrics (e.g. `mae()`) and time-dependent stepped metrics (e.g. `ae()`); detailed breakdowns go to `metrics_per_series.json` |
| `model.backtest(...)` | Windowed-, per-horizon-, or aggregated metrics prefixed with `backtest_`; with `log_backtest_aggregate=True` also logs `backtest_agg_{metric}` |
| PyTorch models (NBEATS, TFT, …) | Per-epoch `train_loss` / `val_loss` via MLflow's PyTorch autolog |

Disable anytime with `autolog(disable=True)`.

For manual logging, save/load APIs, and metric-key details, see the [`Darts API reference`](https://unit8co.github.io/darts/generated_api/darts.utils.mlflow.html) and the [`MLflow API reference`](https://mlflow.org/docs/latest/ml/tracking/tracking-api/).

</details>

## 4. Compare model experiments

We train three comparable sklearn-based models inside separate MLflow runs. Each run follows the same evaluation recipe:

1. **Fit** (pre-train) on the training series (params and model artifact logged automatically).
2. **Historical forecasts** over the same validation period (rolling forecasts).
3. **Backtest** with windowed `mae` (`reduction=None` → stepped `backtest_mae` chart + scalar aggregate `backtest_agg_mae`).
4. **Backtest** with time-dependent `err` (per-horizon error profile `backtest_ae` + scalar aggregate `backtest_agg_ae`).

In [6]:
def run_experiment(run_name, model, *, plot_historical=False):
    """Fit, evaluate, and autolog a single forecasting experiment."""
    hfc_kwargs = {
        "series": series,
        "start": val.start_time(),  # start time of the validation period
        "forecast_horizon": FORECAST_HORIZON,  # forecast horizon
        "stride": BACKTEST_STRIDE,  # step size between rolling forecasts
        "retrain": False,  # use pre-trained model
        "last_points_only": False,  # use all available points for each forecast
    }

    with mlflow.start_run(run_name=run_name):
        # log pre-trained model artifact
        model.fit(train)

        hfcs = model.historical_forecasts(**hfc_kwargs)
        bt_kwargs = {**hfc_kwargs, "historical_forecasts": hfcs, "reduction": None}

        # windowed aggregate metric: stepped backtest_mae + scalar backtest_agg_mae
        model.backtest(metric=metrics.mae, **bt_kwargs)

        # time-dependent metric: per-horizon error `backtest_err` + scalar backtest_agg_err
        model.backtest(metric=metrics.err, **bt_kwargs)

        if plot_historical:
            fig = series.plotly(label="actual")
            for idx, hfc in enumerate(hfcs):
                fig = hfc.plotly(label=f"forecast {idx}", fig=fig)
            fig.update_layout(
                title=f"Historical forecasts — {run_name}", **PLOTLY_KWARGS
            )
            fig.show()

    return model

In [7]:
experiments = [
    (
        "linear_regression",
        LinearRegressionModel(lags=12, output_chunk_length=FORECAST_HORIZON),
        True,
    ),
    (
        "random_forest",
        RandomForestModel(
            lags=12,
            output_chunk_length=FORECAST_HORIZON,
            n_estimators=50,
            random_state=42,
        ),
        False,
    ),
    (
        "linear_regression_lags24",
        LinearRegressionModel(lags=24, output_chunk_length=FORECAST_HORIZON),
        False,
    ),
]

for run_name, model, plot_hist in experiments:
    run_experiment(run_name, model, plot_historical=plot_hist)
    print(f"Finished run: {run_name}")

# disable autologging
autolog(disable=True)

2026/09/03 16:10:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/09/03 16:10:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Finished run: linear_regression
Finished run: random_forest


2026/09/03 16:10:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Finished run: linear_regression_lags24


### Explore in the MLflow UI

Open the tracking UI and compare runs side by side — model parameters, metrics, model artifacts are all available automatically.

In [8]:
print(
    f"\nTo explore runs in the UI:\n  mlflow ui --backend-store-uri sqlite:///{mlflow_db}"
)


To explore runs in the UI:
  mlflow ui --backend-store-uri sqlite:////var/folders/4h/l09drklx06g8022q7khgyxyr0000gn/T/tmps5wn0e4a/mlflow.db


<details open>
<summary><strong>MLflow Runs Table View</strong></summary>

The MLflow Runs Table shows all runs under an experiment in tabular style. It allows you to filter, sort, compare anything that was logged. You can add any logged parameter (metrics, model parameters, ...) to the table via the `Columns` dropdown.

Here we see that `linear_regression` had the lowest aggregated MAE (`backtest_agg_mae`) and aggregated Horizon-Error (Bias) (`backtest_agg_err`) across all runs.

![MLflow runs table view](./static/images/mlflow_experiments_overview.png)

</details>

<details open>
<summary><strong>MLflow Runs Chart View</strong></summary>

Click on the `Chart View` icon on the top left of the Runs Table to show a detailed view of all metrics across runs. In this view you can find scalar as well as stepped metric charts.

- 📈 Stepped metrics (line charts): 
  - Windowed-backtest results for [time-aggregated metrics](https://unit8co.github.io/darts/generated_api/darts.metrics.html) - showing the score per rolling forecast (0, 1, ... number of rolling windows - 1). See `backtest_mae` below showing 3 steps corresponding to the 3 rolling historical forecasts. 
  - Horizon-based-backtest results for [time-dependent metrics](https://unit8co.github.io/darts/generated_api/darts.metrics.html) - showing the score per step in the forecast horizon (0, 1, ... horizon - 1) aggregated over all rolling forecasts. See `backtest_err` below showing 12 steps corresponding to `FORECAST_HORIZON`.
- 📊 Scalar metrics (bar charts) for: 
  - Aggregated backtest metrics, e.g. with `autolog(log_backtest_aggregate=True)`. See `backtest_agg_mae` and `backtest_agg_ae` below.
  - Direct [time-aggregated metric](https://unit8co.github.io/darts/generated_api/darts.metrics.html) calls (e.g. `darts.metrics.mae()`)

... and many other configurations. Read more in the [API reference](https://unit8co.github.io/darts/generated_api/darts.utils.mlflow.html).

![MLflow runs chart view](./static/images/mlflow_experiments_metrics.png)

</details>

<details>
<summary><strong>MLflow Run Detail Page</strong> (click to expand)</summary>

Click on any run in the Runs Table to open the Run Detail Page. It shows an overview of the run, its recorded metrics, hyper-parameters, tags, and more. Play around with the UI to see the different views and features.

Scroll down to the "Model" section and you will see the model that was logged during training. Click on the model to view the details (inclduing the logged model artifacts).

![MLflow run detail page](./static/images/mlflow_run_detail.png)

</details>

## 5. Find the best run

Above, we could already see that `linear_regression` was the best performing model overall.

Let's also do it programatically by finding the model with the lowest MAE. With `log_backtest_aggregate=True` we get a convient sort key: one scalar that summarizes rolling backtest performance, regardless of how many windows or time series were evaluated. `backtest_agg_mae` is such a key.

In [9]:
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.backtest_agg_mae ASC"],
)
display(runs_df[["run_id", "tags.mlflow.runName", "metrics.backtest_agg_mae"]])

best_run = runs_df.iloc[0]
best_run_id = best_run.run_id
best_run_name = best_run["tags.mlflow.runName"]
print(f"\nBest run by backtest_agg_mae: {best_run_name} ({best_run_id})")

,run_id,tags.mlflow.runName,metrics.backtest_agg_mae
0,fb3ad151ba8c4192bb82857a5e255994,linear_regression,18.870433
1,41484b2e9f794a3a96ee6f89f4d789e4,linear_regression_lags24,24.509846
2,2302431c94a348d09dcd46bbc10dcdab,random_forest,77.316667



Best run by backtest_agg_mae: linear_regression (fb3ad151ba8c4192bb82857a5e255994)


## 6. Register the champion model

We can promote the best run's model to the **MLflow Model Registry**, then assign a **champion** alias (or any other alias). Downstream code can then load the registered models via registered model name and alias.

> This assumes that model logging was enabled: `autolog(log_models=True)` and `fit()` was called within each MLflow run

In [10]:
# Get the logged model URI of the best run
model_outputs = mlflow.get_run(best_run_id).outputs.model_outputs
best_model_uri = f"models:/{model_outputs[0].model_id}"
print(f"Best model URI: {best_model_uri}")

REGISTERED_MODEL_NAME = "darts-air-passengers-forecaster"

registration = mlflow.register_model(
    model_uri=best_model_uri,
    name=REGISTERED_MODEL_NAME,
)

client = MlflowClient()
client.set_registered_model_alias(
    REGISTERED_MODEL_NAME, "champion", registration.version
)
print("Aliases set: @champion")

Best model URI: models:/m-d142c7979f29495d90aa74b3e91638e0
Aliases set: @champion


Successfully registered model 'darts-air-passengers-forecaster'.
Created version '1' of model 'darts-air-passengers-forecaster'.


All logged models can be found in the MLflow Models View:

![MLflow Model Table](./static/images/mlflow_models_overview.png)

The registered models can be found in the MLflow Model Registry:

![MLflow Model Registry](./static/images/mlflow_model_registry.png)

## 7. Load the champion and forecast

Production inference loads the aliased registered model via `models:/<name>@champion` without hard-coding a version number.
The loaded model can then be used for prediction:

- Global models (like the sklearn regressors used here) need `series=` at predict time because the model is saved without training data.
- Local models (like `ExponentialSmoothing` and others) must be re-fit before prediction.

> Always load Darts models with `from darts.utils.mlflow import load_model` — not `mlflow.pyfunc.load_model`.

In [11]:
champion = load_model(f"models:/{REGISTERED_MODEL_NAME}@champion")

if isinstance(champion, GlobalForecastingModel):
    forecast = champion.predict(n=FORECAST_HORIZON, series=series)
else:
    forecast = champion.fit(series).predict(n=FORECAST_HORIZON)

fig = series.plotly(label="full series")
fig = forecast.plotly(label="champion forecast", fig=fig)
fig.update_layout(
    title=f"Production inference — {REGISTERED_MODEL_NAME}@champion", **PLOTLY_KWARGS
)

## Summary

In a few lines of code you get a complete experimentation loop:

1. **Setup** — tracking URI + experiment.
2. **Autolog** — `autolog(log_models=True, log_backtest_aggregate=True)`.
3. **Experiment** — `with mlflow.start_run()`: fit, historical forecasts, backtest metrics, hold-out MAPE .
4. **Compare** — `mlflow.search_runs(..., order_by=["metrics.backtest_agg_mae ASC"])`.
5. **Promote** — `register_model` + `set_registered_model_alias(..., "champion", ...)`.
6. **Serve** — `load_model("models:/<name>@champion").predict(...)`.

**Learn more:**

- [Darts MLflow API reference](https://unit8co.github.io/darts/generated_api/darts.utils.mlflow.html)
- [MLflow API reference](https://mlflow.org/docs/latest/ml/tracking/tracking-api/)
